# Syntax Parsing

In this lab we will see how to use NLTK for PCFGs

## 1. Constituent-based syntactic parsing

First of all, we will need to import nltk as usual and the nltk corpus of parsed WSJ treebank. This is a small portion of the WSJ treebank, but it will be enough for our purposes.

In [2]:
import nltk
nltk.download('treebank')
from nltk.corpus import treebank

print(treebank.parsed_sents('wsj_0001.mrg')[0])

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\Alexander\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\treebank.zip.


(S
  (NP-SBJ
    (NP (NNP Pierre) (NNP Vinken))
    (, ,)
    (ADJP (NP (CD 61) (NNS years)) (JJ old))
    (, ,))
  (VP
    (MD will)
    (VP
      (VB join)
      (NP (DT the) (NN board))
      (PP-CLR (IN as) (NP (DT a) (JJ nonexecutive) (NN director)))
      (NP-TMP (NNP Nov.) (CD 29))))
  (. .))


You can also use the **draw()** method to obtain a graphical representation of the parse tree:

In [3]:
treebank.parsed_sents('wsj_0001.mrg')[0].draw()

This method is part of the Tree NLTK class and has methods to read parse trees in bracketed format:

In [4]:
from nltk import Tree

s = '(S (NP (DT the) (NN cat)) (VP (VBD ate) (NP (DT a) (NN cookie))))'
t = Tree.fromstring(s)
print(t)


(S (NP (DT the) (NN cat)) (VP (VBD ate) (NP (DT a) (NN cookie))))


Let's see some other useful methods for the **Tree** class:

- Transforming the tree in Chomsky’s Normal Form:

In [5]:
t.chomsky_normal_form()

Obtaining the productions from a parse tree:

In [6]:
t.productions()

[S -> NP VP,
 NP -> DT NN,
 DT -> 'the',
 NN -> 'cat',
 VP -> VBD NP,
 VBD -> 'ate',
 NP -> DT NN,
 DT -> 'a',
 NN -> 'cookie']

For every production we can get the left and right parts of the productions:

In [7]:
for p in t.productions():
    print (p.lhs()) #left part (nonterminal)
    print (p.rhs()) #right part of the rule (sequence)

S
(NP, VP)
NP
(DT, NN)
DT
('the',)
NN
('cat',)
VP
(VBD, NP)
VBD
('ate',)
NP
(DT, NN)
DT
('a',)
NN
('cookie',)


### 1.1 Context-Free Grammars

NLTK includes several classes to encode CFG and PCFG grammars in the nltk.grammar module:

1.	Nonterminal
2.	Production (for rules)
3.	WeightedProduction (for rules in a PCFG)
4.	ContextFreeGrammar
5.	PCFG

The classes include convenient parsers to convert strings into grammars. The method *grammar.check_coverage(ws)* determines whether the words that appear in *ws* can ever be parsed by the grammar.

In [8]:
from nltk import nonterminals, Nonterminal, Production

# Create some nonterminals
S, NP, VP, PP, VB, NN = nonterminals('S, NP, VP, PP, VB, NN')
N, V, P, Det = nonterminals('N, V, P, Det')
# Create a production rule and print it
print(Production(S, [NP,VP]))


S -> NP VP


**Exercise 1**. Using the NLTK classes to encode elements of CFG grammars, estimate the probability of observing the rule 'NP -> NP PP' and the rule 'VP -> VB NP'

Remember that the probability for productions is estimated as:

$$p( X \to \alpha) =  \frac{c( X \to \alpha )}{c(X)}$$

In [9]:
#Suggestion: the following code helps browsing the treebank

productions=[]
for item in treebank.fileids():
    for tree in treebank.parsed_sents(item):
        tree.chomsky_normal_form() # Don’t forget to normalize in CNF the parse tree
        productions += tree.productions()


In [14]:
from nltk import Production
from collections import Counter
# Define the nonterminals
S, NP, VP, PP, VB, NN = nonterminals('S, NP, VP, PP, VB, NN')

# Count the occurrences of the productions
production_counts = Counter(productions)

# Calculate the probabilities
np_pp_count = production_counts[Production(NP, [NP, PP])]
vp_vb_np_count = production_counts[Production(VP, [VB, NP])]

total_np_productions = sum(count for prod, count in production_counts.items() if prod.lhs() == NP)
total_vp_productions = sum(count for prod, count in production_counts.items() if prod.lhs() == VP)

prob_np_pp = np_pp_count / total_np_productions if total_np_productions > 0 else 0
prob_vp_vb_np = vp_vb_np_count / total_vp_productions if total_vp_productions > 0 else 0
print(f"Probability of 'NP -> NP PP': {prob_np_pp:.4f}")
print(f"Probability of 'VP -> VB NP': {prob_vp_vb_np:.4f}")

Probability of 'NP -> NP PP': 0.0922
Probability of 'VP -> VB NP': 0.0555


### 1.2 Moving to Probabilistic Context-Free Grammars

NLTK can do the work of inducing a PCFG from an existing Treebank. Once we have stored in productions all the productions from our treebank, we can obtain the PCFG as follows:

In [12]:
from nltk import Nonterminal

S = Nonterminal('S')
grammar = nltk.grammar.induce_pcfg(S, productions)

**Exercise 2**. Compare the values obtained in exercise 1 to those calculated automatically by the induce_pcfg method. (Hint: grammar.productions() to navigate through the productions of the grammar).

In [13]:
nonter = NP
seq = (NP, PP)

for p in grammar.productions():
    if p.lhs()== nonter:
        if p.rhs()== seq:
            print(p)
            

NP -> NP PP [0.0922273]


==> I have the same one


### 1.3 Parsing

There are different types of parsers implemented in NLTK. One that implements the Viterbi CKY n-best parses over a PCFG is available in the *parse.viterbi* module:

In [15]:
for p in grammar.productions():
    if p.lhs()== NN:
        if p.rhs()[0].startswith("tel"):
            print(p)

NN -> 'television' [0.000683579]
NN -> 'telephone' [0.000759532]
NN -> 'telecommunications' [7.59532e-05]
NN -> 'telegraph' [7.59532e-05]
NN -> 'telephone-information' [7.59532e-05]


In [16]:
from nltk.parse import ViterbiParser

s='I saw John with my telephone'
tokens = s.split() #simplified – you can use a tokenizer
parser=ViterbiParser(grammar) #using the grammar induced by the Treebank
parses = parser.parse_all(tokens) #the resulting parse tree


**Exercise 3**. Test the above sentence and draw the resulting parse. Is this parse tree correct?
Verify if the sentence "I saw John with my telescope" is parsable.

In [17]:
parses[0].draw()